# FIM-ODE: Evaluation on Reference Datasets

> **Notebook focus:** Single-trajectory evaluation on the Hegde et al. benchmark datasets.  
> All helper functions live in `scripts/ode/fim_ode.py`; this notebook orchestrates loading,
> preprocessing, evaluation, and reporting.

## 1. Imports & device

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

# ── Locate repo root and add experiments/ to path ────────────────────
_nb_path   = Path(globals().get('__vsc_ipynb_file__',
                  __file__ if '__file__' in dir() else __import__('os').getcwd()))
_repo_root = _nb_path.parent
while not (_repo_root / 'scripts' / 'ode').exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent

if str(_repo_root / 'scripts' / 'ode') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'scripts' / 'ode'))

# Reload in case the module was already imported in this kernel session
import importlib, fim_ode as _fim_ode_mod
importlib.reload(_fim_ode_mod)

from fim_ode import (
    # ODE dynamics
    vdp_dynamics, fhn_dynamics,
    # Preprocessing
    preprocess_ref_vdp_uniform, preprocess_ref_vdp_nonuniform, preprocess_ref_fhn,
    # FIM-ODE evaluation
    eval_ref_vdp, eval_ref_fhn,
    # ODEFormer
    load_odeformer, eval_ref_vdp_odeformer, eval_ref_fhn_odeformer,
    # Reporting
    stats_to_markdown, stats_to_latex, save_tables,
    # Plotting
    plot_ref_evaluation,
)

# MPS is missing some ops this model uses (cummax), so we run on CPU
device = 'cpu'
print(f'Device: {device}')

## 2. Load the pretrained model from HuggingFace

Weights are cached locally after the first download.

In [ ]:
import json
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from fim.models.fim_ode import FIMODE as FimOdeonUnified
from fim.models.fim_ode_trainer import FIMODEConfig as TrainingWrapperConfiguration

REPO_ID   = 'FIM4Science/fim-ode'
SUBFOLDER = 'base_model/checkpoints/best-model'

config_path  = hf_hub_download(REPO_ID, f'{SUBFOLDER}/config.json')
weights_path = hf_hub_download(REPO_ID, f'{SUBFOLDER}/model.safetensors')
print('Downloaded to:', weights_path)

with open(config_path) as f:
    config_dict = json.load(f)
config = TrainingWrapperConfiguration()
config.model_config = config_dict['model_config']
config.train_config = config_dict['train_config']

model = FimOdeonUnified(config)
state_dict = load_file(weights_path, device=device)
state_dict = {k.removeprefix('model.'): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model = model.to(device).eval()
print('Model loaded successfully')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Reference datasets

Pre-generated benchmark datasets from Hegde et al. (UAGP-ODE).  
Stored in `scripts/ode/hedge_gp_odes_data/`.

In [ ]:
data_dir = _repo_root / 'scripts' / 'ode' / 'hedge_gp_odes_data'
print('data_dir:', data_dir.resolve())
assert data_dir.exists(), f'Not found: {data_dir}'

ref_vdp_u = np.load(data_dir / 'vdp_uniform.npz')
ref_vdp_n = np.load(data_dir / 'vdp_nonuniform.npz')
ref_fhn   = np.load(data_dir / 'fhn_interpolation.npz')

# ── Print shapes ─────────────────────────────────────────────────────
for name, d in [('VDP-uniform', ref_vdp_u), ('VDP-nonuniform', ref_vdp_n), ('FHN', ref_fhn)]:
    print(f'\n── {name} ──')
    for k, v in sorted(d.items()):
        info = f'shape={v.shape}  dtype={v.dtype}' if hasattr(v, 'shape') and v.ndim > 0 else f'value={v}'
        print(f'  {k:25s}: {info}')

## 4. Preprocess reference datasets

Three fixes are needed before evaluation:

- **VDP-uniform** `test_ts/test_ys` cover the full `[0, 14]` window → keep only the **last 50** points (`[7, 14]`).
- **VDP-nonuniform** `test_ts/test_ys` have **100** points on `[7, 14]` → randomly subsample to **50**.
- **FHN** context is split into contiguous observed sub-trajectories so the model never creates
  transition features that straddle the missing region (x₁ > 0 & x₂ < 0).

In [ ]:
# ── VDP-uniform: keep last 50 test points ──────────────────────────────
ref_vdp_u_test_ts, ref_vdp_u_test_ys = preprocess_ref_vdp_uniform(ref_vdp_u)
print(f'VDP-u  test_ts: [{ref_vdp_u_test_ts[0]:.3f}, {ref_vdp_u_test_ts[-1]:.3f}]  ({len(ref_vdp_u_test_ts)} pts)')

# ── VDP-nonuniform: subsample 50 of 100 test points ────────────────────
ref_vdp_n_test_ts, ref_vdp_n_test_ys = preprocess_ref_vdp_nonuniform(ref_vdp_n, seed=0)
print(f'VDP-n  test_ts: [{ref_vdp_n_test_ts[0]:.3f}, {ref_vdp_n_test_ts[-1]:.3f}]  ({len(ref_vdp_n_test_ts)} pts)')

# ── FHN: split at missing-region boundaries, build padded context ───────
(
    fhn_ctx_trajs, fhn_ctx_times, fhn_ctx_mask,
    fhn_ref_miss_mask, fhn_ref_target_ts, fhn_ref_target_ys,
) = preprocess_ref_fhn(ref_fhn)
print(f'FHN    context: {fhn_ctx_trajs.shape[0]} segments, '
      f'max_len={fhn_ctx_trajs.shape[1]},  '
      f'{int(fhn_ctx_mask.sum())} observed pts,  '
      f'{int(fhn_ref_miss_mask.sum())} missing pts')

## 5. ODEFormer evaluation

ODEFormer (sdascoli/odeformer) performs symbolic regression on ODE trajectories:
given the training observations it searches for a closed-form symbolic ODE that
fits the data, then integrates it numerically from the known initial condition.

We use the same reference datasets and the same evaluation protocol as FIM-ODE:
integration starts from the true IC at `t=0`, MSE is measured on the test window.
For FHN the model is fitted on the observed time steps only (outside the missing
quadrant), then integrated from `x0` over the full grid.

In [ ]:
# Compatibility patches (np.infty + torch.load) are applied automatically
# when fim_ode is imported/reloaded in cell s1_code — nothing extra needed here.

from fim_ode import load_odeformer
odeformer = load_odeformer(device='cpu', beam_size=50, beam_temperature=0.1)

In [ ]:
print('=== ODEFormer — Reference dataset evaluation ===\n')

pred_od_vdp_u, t_od_full_u, mse_od_vdp_u = eval_ref_vdp_odeformer(
    odeformer,
    ref_vdp_u['train_ts'], ref_vdp_u['train_ys'], ref_vdp_u['x0'],
    ref_vdp_u_test_ts, ref_vdp_u_test_ys, 'VDP-uniform',
)

pred_od_vdp_n, t_od_full_n, mse_od_vdp_n = eval_ref_vdp_odeformer(
    odeformer,
    ref_vdp_n['train_ts'], ref_vdp_n['train_ys'], ref_vdp_n['x0'],
    ref_vdp_n_test_ts, ref_vdp_n_test_ys, 'VDP-nonuniform',
)

# FHN: fit on observed steps only
_full_ts = ref_fhn['full_ts']
_full_ys = ref_fhn['full_ys'][0]      # (T, D)
_obs_mask = ~fhn_ref_miss_mask         # True = observed

pred_od_fhn, mse_od_fhn = eval_ref_fhn_odeformer(
    odeformer,
    _full_ts, _full_ys, _obs_mask,
    ref_fhn['x0'], fhn_ref_miss_mask, fhn_ref_target_ys,
)

## 6. FIM-ODE evaluation

Single-trajectory evaluation: for each system the model receives **one** context trajectory
(the training observations) and must predict the held-out region.

In [ ]:
print('=== FIM-ODE — Reference dataset evaluation ===\n')

pred_ref_vdp_u, t_ref_full_u, mse_ref_vdp_u = eval_ref_vdp(
    model, device,
    ref_vdp_u['train_ts'], ref_vdp_u['train_ys'], ref_vdp_u['x0'],
    ref_vdp_u_test_ts, ref_vdp_u_test_ys, 'VDP-uniform',
)

pred_ref_vdp_n, t_ref_full_n, mse_ref_vdp_n = eval_ref_vdp(
    model, device,
    ref_vdp_n['train_ts'], ref_vdp_n['train_ys'], ref_vdp_n['x0'],
    ref_vdp_n_test_ts, ref_vdp_n_test_ys, 'VDP-nonuniform',
)

pred_ref_fhn, mse_ref_fhn = eval_ref_fhn(
    model, device,
    fhn_ctx_trajs, fhn_ctx_times, fhn_ctx_mask,
    ref_fhn['full_ts'], ref_fhn['x0'],
    fhn_ref_miss_mask, fhn_ref_target_ys,
)

### 6.1 FIM-ODE — Finetuned models

Load the best finetuned checkpoint for each reference system, evaluate on the
same held-out test sets as section 6, then release the model from memory.

| System | Checkpoint | Best epoch | Test MSE |
|---|---|---|---|
| VDP-uniform | `vdp-u_20260525_184055_best.pt` | 90 | 0.02836 |
| VDP-nonuniform | `vdp-nu_20260525_200128_best.pt` | 55 | 0.02230 |
| FHN | `fhn_20260525_190513_best.pt` | 180 | 0.02975 |

#### Context format for FHN

The zero-shot evaluation above (section 6) uses `preprocess_ref_fhn`, which splits
the 38 observed points into **two separate segments** around the missing region and
pads them to a uniform length. This preprocessing was necessary for the zero-shot
model because FIM-ODE internally computes consecutive differences (Δy, Δt); without
the split, the large time jump at the gap boundary would produce a spurious transition
that corrupts the encoded vector field.

For the **finetuned** model we skip that preprocessing and pass all 38 observed points
as a **single flat trajectory** — exactly the format seen during finetuning
(`mode: full_trajectory` in the config). The model has been adapted to this context
format, so presenting the gap as a large-but-real Δt is consistent with how it was
trained. The integration and MSE computation are otherwise identical: RK45 from x0
over the full time grid, MSE evaluated only at the 12 missing interpolation points.

In [ ]:
import gc
import numpy as np
import torch
from fim.models.fim_ode import FIMODE as FimOdeonUnified

_ckpt_dir = _repo_root / 'scripts' / 'ode' / 'checkpoints' / 'finetune'

# Reuse config already loaded in cell 2
_ft_model = FimOdeonUnified(config).to(device)

print('=== FIM-ODE (finetuned) — Reference dataset evaluation ===\n')

# ── VDP-uniform ──────────────────────────────────────────────────────
_ckpt = torch.load(_ckpt_dir / 'vdp-u_20260525_184055_best.pt', map_location=device, weights_only=False)
_ft_model.load_state_dict(_ckpt['model_state_dict'])
_ft_model.eval()
_, _, mse_ft_vdp_u = eval_ref_vdp(
    _ft_model, device,
    ref_vdp_u['train_ts'], ref_vdp_u['train_ys'], ref_vdp_u['x0'],
    ref_vdp_u_test_ts, ref_vdp_u_test_ys, 'VDP-uniform (finetuned)',
)

# ── VDP-nonuniform ───────────────────────────────────────────────────
_ckpt = torch.load(_ckpt_dir / 'vdp-nu_20260525_200128_best.pt', map_location=device, weights_only=False)
_ft_model.load_state_dict(_ckpt['model_state_dict'])
_ft_model.eval()
_, _, mse_ft_vdp_n = eval_ref_vdp(
    _ft_model, device,
    ref_vdp_n['train_ts'], ref_vdp_n['train_ys'], ref_vdp_n['x0'],
    ref_vdp_n_test_ts, ref_vdp_n_test_ys, 'VDP-nonuniform (finetuned)',
)

# ── FHN ──────────────────────────────────────────────────────────────
# Pass the full flat trajectory (no segment splitting) — consistent with
# how the model was finetuned. See the markdown cell above for rationale.
_fhn_flat_trajs = ref_fhn['train_ys']                                # (1, 38, 2)
_fhn_flat_times = ref_fhn['train_ts'][np.newaxis]                    # (1, 38)
_fhn_flat_mask  = np.ones((1, len(ref_fhn['train_ts'])), dtype=bool) # (1, 38) all observed

_ckpt = torch.load(_ckpt_dir / 'fhn_20260525_190513_best.pt', map_location=device, weights_only=False)
_ft_model.load_state_dict(_ckpt['model_state_dict'])
_ft_model.eval()
_, mse_ft_fhn = eval_ref_fhn(
    _ft_model, device,
    _fhn_flat_trajs, _fhn_flat_times, _fhn_flat_mask,
    ref_fhn['full_ts'], ref_fhn['x0'],
    fhn_ref_miss_mask, fhn_ref_target_ys,
    label='FHN (missing quadrant, finetuned)',
)

# ── Release from memory ───────────────────────────────────────────────
del _ft_model, _ckpt
gc.collect()

# ── Comparison table ─────────────────────────────────────────────────
print(f'\n  {"System":<20} {"Zero-shot":>12} {"Finetuned":>12} {"Improvement":>13}')
print('  ' + '-' * 59)
for name, mse0, mse1 in [
    ('VDP-uniform',    mse_ref_vdp_u, mse_ft_vdp_u),
    ('VDP-nonuniform', mse_ref_vdp_n, mse_ft_vdp_n),
    ('FHN',            mse_ref_fhn,   mse_ft_fhn),
]:
    pct = 100 * (mse0 - mse1) / mse0
    print(f'  {name:<20} {mse0:>12.5f} {mse1:>12.5f} {pct:>12.1f}%')

## 7. Visualisation

In [ ]:
fig = plot_ref_evaluation(
    ref_vdp_u, ref_vdp_n, ref_fhn,
    pred_ref_vdp_u, t_ref_full_u, mse_ref_vdp_u,
    pred_ref_vdp_n, t_ref_full_n, mse_ref_vdp_n,
    pred_ref_fhn,   mse_ref_fhn,
    ref_vdp_u_test_ts, ref_vdp_u_test_ys,
    ref_vdp_n_test_ts, ref_vdp_n_test_ys,
    fhn_ref_miss_mask, fhn_ref_target_ts, fhn_ref_target_ys,
)
plt.show()

## 8. Comparison table

MSE for FIM-ODE and ODEFormer on the three reference datasets.

In [ ]:
# ── Console comparison table ────────────────────────────────────────────
rows = [
    ('VDP-uniform',    mse_ref_vdp_u, mse_od_vdp_u),
    ('VDP-nonuniform', mse_ref_vdp_n, mse_od_vdp_n),
    ('FHN',            mse_ref_fhn,   mse_od_fhn),
]

print(f'\n  {"System":<20s} {"FIM-ODE":>10} {"ODEFormer":>10}')
print(f'  {"-"*42}')
for sys_name, fim_mse, od_mse in rows:
    od_str = f'{od_mse:10.5f}' if not (od_mse != od_mse) else '     FAIL'
    print(f'  {sys_name:<20s} {fim_mse:10.5f} {od_str}')

## 9. Save tables

Use `save_tables` from `fim_ode.py` to write Markdown or LaTeX output.

In [ ]:
output_dir = _repo_root / 'results' / 'fim-ode'
output_dir.mkdir(parents=True, exist_ok=True)

# ── Markdown ─────────────────────────────────────────────────────────
def _fmt(v):
    return f'{v:.5f}' if v == v else 'FAIL'  # NaN check

md_lines = [
    '# FIM-ODE — Reference Dataset Evaluation\n',
    '| System | FIM-ODE | ODEFormer |',
    '|--------|--------:|----------:|',
    f'| VDP-uniform    | {_fmt(mse_ref_vdp_u)} | {_fmt(mse_od_vdp_u)} |',
    f'| VDP-nonuniform | {_fmt(mse_ref_vdp_n)} | {_fmt(mse_od_vdp_n)} |',
    f'| FHN            | {_fmt(mse_ref_fhn)}   | {_fmt(mse_od_fhn)}   |',
]
md_path = output_dir / 'ref_evaluation.md'
md_path.write_text('\n'.join(md_lines) + '\n')
print(f'Saved: {md_path}')

# ── LaTeX ────────────────────────────────────────────────────────────
tex_lines = [
    r'\begin{table}[ht]',
    r'\centering',
    r'\caption{MSE on Hegde et al.\ reference datasets}',
    r'\label{tab:ref_eval}',
    r'\begin{tabular}{lrr}',
    r'\toprule',
    r'System & FIM-ODE & ODEFormer \\',
    r'\midrule',
    f'VDP-uniform    & {_fmt(mse_ref_vdp_u)} & {_fmt(mse_od_vdp_u)} \\\\',
    f'VDP-nonuniform & {_fmt(mse_ref_vdp_n)} & {_fmt(mse_od_vdp_n)} \\\\',
    f'FHN            & {_fmt(mse_ref_fhn)}   & {_fmt(mse_od_fhn)}   \\\\',
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table}',
]
tex_path = output_dir / 'ref_evaluation.tex'
tex_path.write_text('\n'.join(tex_lines) + '\n')
print(f'Saved: {tex_path}')

---
## 10. ODEBench: Fixed-Point & Stability Analysis

We evaluate FIM-ODE and ODEFormer on three systems from the ODEBench dataset:
- **System 37** — Van der Pol oscillator (VDP): $\dot x_0 = x_1,\; \dot x_1 = -x_0 - c_0(x_0^2-1)x_1$, $c_0=0.43$
- **System 40** — Duffing oscillator: $\dot x_0 = x_1,\; \dot x_1 = -x_0 + c_0 x_1(1-x_0^2)$, $c_0=0.886$
- **System 42** — Reduced CDIMA model (chlorine dioxide–iodine–malonic acid), 2D chemical oscillator

For each system we:
1. Load ODEBench data with multiplicative noise + random subsampling
2. Fit both models to the context trajectories
3. Compute **ground-truth** fixed points and Jacobians symbolically via SymPy
4. Find **numerical** fixed points for FIM-ODE and ODEFormer via multi-start `fsolve`
5. Evaluate the Jacobian at each fixed point and classify stability
6. Plot phase-space portraits: Ground Truth | FIM-ODE | ODEFormer

In [ ]:
import importlib, fim_ode
importlib.reload(fim_ode)
from fim_ode import (
    load_odebench_system, make_fim_vf_fn, build_odeformer_vf_fn,
    find_fixed_points, numerical_jacobian, symbolic_jacobian,
    stability_analysis,
)
import numpy as np

SEED = 0
SYSTEM_IDS   = [26, 42, 28]
SYSTEM_NAMES = {26: 'LVComp', 42: 'CDIMA', 28: 'pendulum'}

# Load systems: only the first trajectory used as context (index 0)
systems = {}
for sid in SYSTEM_IDS:
    times, trajs, meta = load_odebench_system(
        sid, noise_sigma=0.03, subsample_fraction=0.5,
        rng=np.random.default_rng(SEED),
    )
    # [:1] keeps shape (1, T, D) / (1, T, 1) — needed by make_fim_vf_fn
    systems[sid] = dict(times=times[:1], trajs=trajs[:1], meta=meta)
    print(f"System {sid} ({SYSTEM_NAMES[sid]}): eq = {meta['eq']}")
    print(f"  using trajs {trajs[:1].shape}, times {times[:1].shape}, consts {meta['consts']}")

In [ ]:
import sympy as sp


def sympy_analysis(eq_str, const_subs, label=''):
    """Symbolic fixed points, Jacobian, and stability via SymPy."""
    eq_str = eq_str.replace('^', '**')
    parts  = [p.strip() for p in eq_str.split('|')]
    D      = len(parts)
    xs     = [sp.Symbol(f'x_{i}') for i in range(D)]
    cs     = {sp.Symbol(f'c_{i}'): v for i, v in enumerate(const_subs)}

    exprs = [sp.sympify(p).subs(cs) for p in parts]
    J_sym = sp.Matrix([[sp.diff(e, x) for x in xs] for e in exprs])

    try:
        fps_sym = sp.solve(exprs, xs, dict=True)
    except Exception:
        fps_sym = []

    results = []
    print(f'\n-- {label} --')
    print(f'  f(x) = {exprs}')
    for fp_dict in fps_sym:
        fp_num = np.array([complex(fp_dict.get(x, 0)) for x in xs])
        if np.any(np.abs(fp_num.imag) > 1e-8):
            print(f'  Fixed point (complex, skipped): {fp_dict}')
            continue
        fp_num = fp_num.real
        J_num  = np.array(J_sym.subs(fp_dict).evalf().tolist(), dtype=float)
        s      = stability_analysis(J_num)
        results.append(dict(fp=fp_num, J=J_num, **s))
        print(f'  Fixed point : {np.round(fp_num, 4)}')
        print(f'  Jacobian    :\n{np.round(J_num, 4)}')
        print(f'  Eigenvalues : {np.round(s["eigenvalues"], 4)}')
        print(f'  Stability   : {s["label"]}')
    if not results:
        print('  No real fixed points found symbolically.')
    return results


gt_lv   = sympy_analysis(systems[26]['meta']['eq'], systems[26]['meta']['consts'],
                           label='Ground Truth -- LVComp (26)')
gt_cdima = sympy_analysis(systems[42]['meta']['eq'], systems[42]['meta']['consts'],
                           label='Ground Truth -- CDIMA (42)')
gt_pendulum = sympy_analysis(systems[28]['meta']['eq'], systems[28]['meta']['consts'],
                              label='Ground Truth -- pendulum (28)')

In [ ]:
# FIM-ODE: encode each system's context once, find fixed points, compute Jacobians
# via central finite differences (numerical_jacobian).
fim_model = model
DEVICE    = device

fim_results = {}
for sid in SYSTEM_IDS:
    s   = systems[sid]
    vf  = make_fim_vf_fn(fim_model, s['trajs'], s['times'], device=DEVICE)
    fps = find_fixed_points(vf, D=2, n_starts=400, x_range=(-6, 6),
                             rng=np.random.default_rng(SEED), top_k=4)
    name = SYSTEM_NAMES[sid]
    print(f'\n-- FIM-ODE -- {name} ({sid}) --')
    if not fps:
        print('  No fixed points found.')
    for fp, res in fps:
        J      = numerical_jacobian(vf, fp)
        s_info = stability_analysis(J)
        print(f'  Fixed point : {np.round(fp, 4)}   ||f(x*)|| = {res:.2e}')
        print(f'  Jacobian    :\n{np.round(J, 4)}')
        print(f'  Eigenvalues : {np.round(s_info["eigenvalues"], 4)}')
        print(f'  Stability   : {s_info["label"]}')
    fim_results[sid] = dict(vf=vf, fps=fps)

In [ ]:
# ODEFormer: fit per-system, print symbolic equations, find fixed points.
# Jacobian computed two ways:
#   - symbolic  : exact, via SymPy differentiation of the inferred equation
#   - numerical : central finite differences (cross-check)

od_results = {}
for sid in SYSTEM_IDS:
    s    = systems[sid]
    name = SYSTEM_NAMES[sid]
    D    = s['trajs'].shape[-1]

    trajs_f64 = s['trajs'].astype(np.float64)
    times_f64 = s['times'][..., 0].astype(np.float64)

    trs = [trajs_f64[i] for i in range(trajs_f64.shape[0])]
    tms = [times_f64[i] for i in range(times_f64.shape[0])]

    prediction = odeformer.fit(trajectories=trs, times=tms)
    eq_strs    = [str(prediction[i][0]) for i in range(len(prediction))]

    print(f'\n-- ODEFormer -- {name} ({sid}) --')
    print('  Predicted symbolic vector field (best beam):')
    eq_parsed = eq_strs[0].replace('^', '**')
    for dim_i, dim_eq in enumerate(eq_parsed.split('|')):
        print(f'    dx_{dim_i}/dt = {dim_eq.strip()}')

    vf = build_odeformer_vf_fn(eq_parsed, D=D)
    if vf is None:
        print('  Could not parse equation -- skipping fixed-point search.')
        od_results[sid] = dict(eq_strs=eq_strs, eq_parsed=None, vf=None, fps=[])
        continue

    fps = find_fixed_points(vf, D=D, n_starts=400, x_range=(-6, 6),
                             rng=np.random.default_rng(SEED), top_k=4)
    print()
    if not fps:
        print('  No fixed points found.')
    for fp, res in fps:
        J_sym = symbolic_jacobian(eq_parsed, D, fp)
        J_num = numerical_jacobian(vf, fp)
        s_sym = stability_analysis(J_sym)
        s_num = stability_analysis(J_num)
        print(f'  Fixed point          : {np.round(fp, 4)}   ||f(x*)|| = {res:.2e}')
        print(f'  Jacobian (symbolic)  :\n{np.round(J_sym, 4)}')
        print(f'  Eigenvalues          : {np.round(s_sym["eigenvalues"], 4)}  →  {s_sym["label"]}')
        print(f'  Jacobian (numerical) :\n{np.round(J_num, 4)}')
        print(f'  Eigenvalues          : {np.round(s_num["eigenvalues"], 4)}  →  {s_num["label"]}')
        print(f'  Max |J_sym - J_num|  = {np.abs(J_sym - J_num).max():.2e}')
    od_results[sid] = dict(eq_strs=eq_strs, eq_parsed=eq_parsed, vf=vf, fps=fps)

In [ ]:
# Comparison table: fixed points and stability per model vs ground truth
gt_all = {26: gt_lv, 42: gt_cdima, 28: gt_pendulum}

print(f"{'System':<12} {'Model':<12} {'Fixed Point':<22} {'Stability':<24} {'Max Re(lam)':>12} {'||f(x*)||':>12}")
print('-' * 96)

for sid in SYSTEM_IDS:
    name = SYSTEM_NAMES[sid]
    rows = [
        ('GT', gt_all[sid]),
        ('FIM-ODE',
         [dict(fp=fp, res=res, **stability_analysis(numerical_jacobian(fim_results[sid]['vf'], fp)))
          for fp, res in fim_results[sid]['fps']]),
        ('ODEFormer',
         [dict(fp=fp, res=res, **stability_analysis(numerical_jacobian(od_results[sid]['vf'], fp)))
          for fp, res in od_results[sid]['fps']]
         if od_results[sid]['vf'] is not None else []),
    ]
    for model_name, fp_list in rows:
        if not fp_list:
            tag = f"{name} {sid}"
            print(f"{tag:<12} {model_name:<12} {'--':<22} {'--':<24} {'--':>12} {'--':>12}")
            continue
        for i, r in enumerate(fp_list):
            tag    = f"{name} {sid}" if i == 0 else ''
            mname  = model_name      if i == 0 else ''
            fp_str = '[' + ', '.join(f'{v:.3f}' for v in r['fp']) + ']'
            max_re = f"{r['max_real']:+.4f}"
            res_str = f"{r.get('res', float('nan')):.2e}"
            print(f"{tag:<12} {mname:<12} {fp_str:<22} {r['label']:<24} {max_re:>12} {res_str:>12}")
    print()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp


def _make_gt_vf(eq_str, consts, D=2):
    """Build a GT vector-field callable from an ODEBench equation + constant list.

    Substitutes c_0, c_1, … with numerical values via SymPy, then returns a
    numpy-compatible callable ``(N, D) -> (N, D)``.  Works for any system in
    SYSTEM_IDS without hard-coding equations in this cell.
    """
    eq_str = eq_str.replace('^', '**')
    xs = [sp.Symbol(f'x_{i}') for i in range(D)]
    cs = {sp.Symbol(f'c_{i}'): float(v) for i, v in enumerate(consts)}
    parts = [sp.sympify(p.strip()).subs(cs) for p in eq_str.split('|')]
    funcs = [sp.lambdify(xs, expr, modules='numpy') for expr in parts]

    def vf(x):
        cols = [x[:, i] for i in range(D)]
        outs = []
        for f in funcs:
            v = f(*cols)
            if np.isscalar(v):
                v = np.full(x.shape[0], float(v))
            outs.append(np.asarray(v, dtype=float))
        return np.stack(outs, axis=-1)

    return vf


def _traj_range(trajs, margin=0.5, min_pad=1.0):
    """Derive (x0_range, x1_range) from trajectory bounds + padding."""
    tr = trajs[0]   # (T, D)
    ranges = []
    for d in range(tr.shape[-1]):
        lo, hi = float(tr[:, d].min()), float(tr[:, d].max())
        pad = max((hi - lo) * margin, min_pad)
        ranges.append((lo - pad, hi + pad))
    return tuple(ranges)


def phase_portrait(ax, vf, x0_range, x1_range, n_grid=25,
                   fps=None, title='', trajs=None):
    """Quiver phase portrait: normalised direction arrows coloured by speed."""
    x0g = np.linspace(*x0_range, n_grid)
    x1g = np.linspace(*x1_range, n_grid)
    X0, X1 = np.meshgrid(x0g, x1g)
    pts = np.stack([X0.ravel(), X1.ravel()], axis=-1)

    try:
        dX    = vf(pts)
        U = dX[:, 0].reshape(n_grid, n_grid)
        V = dX[:, 1].reshape(n_grid, n_grid)
        speed = np.sqrt(U**2 + V**2) + 1e-10
        ax.quiver(X0, X1, U / speed, V / speed, speed,
                  cmap='viridis', alpha=0.80, scale=30)
    except Exception as e:
        ax.text(0.5, 0.5, f'eval error:\n{e}',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=8, color='red')

    if trajs is not None:
        for tr in trajs:
            ax.plot(tr[:, 0], tr[:, 1], 'w-', lw=0.8, alpha=0.55)

    if fps:
        for fp in fps:
            ax.plot(fp[0], fp[1], 'r*', ms=13, zorder=5,
                    markeredgecolor='k', markeredgewidth=0.5)

    ax.set_xlim(*x0_range)
    ax.set_ylim(*x1_range)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('$x_0$')
    ax.set_ylabel('$x_1$')


# ── Build GT vector fields and plot ranges from loaded system metadata ──────
gt_vfs = {
    sid: _make_gt_vf(systems[sid]['meta']['eq'], systems[sid]['meta']['consts'])
    for sid in SYSTEM_IDS
}
RANGES = {
    sid: _traj_range(systems[sid]['trajs'])
    for sid in SYSTEM_IDS
}

n_sys = len(SYSTEM_IDS)
fig, axes = plt.subplots(n_sys, 3, figsize=(13, 4 * n_sys))
if n_sys == 1:
    axes = axes[np.newaxis]
fig.suptitle('Phase portraits — Ground Truth | FIM-ODE | ODEFormer', fontsize=13)

for row, sid in enumerate(SYSTEM_IDS):
    x0r, x1r = RANGES[sid]
    name      = SYSTEM_NAMES[sid]
    ctx_trajs = systems[sid]['trajs']
    gt_fps    = [r['fp'] for r in gt_all[sid]]

    phase_portrait(axes[row, 0], gt_vfs[sid], x0r, x1r,
                   fps=gt_fps, title=f'{name} — Ground Truth', trajs=ctx_trajs)

    phase_portrait(axes[row, 1], fim_results[sid]['vf'], x0r, x1r,
                   fps=fim_results[sid]['fps'], title=f'{name} — FIM-ODE',
                   trajs=ctx_trajs)

    od_vf  = od_results[sid]['vf']
    od_fps = od_results[sid]['fps']
    if od_vf is not None:
        phase_portrait(axes[row, 2], od_vf, x0r, x1r,
                       fps=od_fps, title=f'{name} — ODEFormer', trajs=ctx_trajs)
    else:
        axes[row, 2].text(0.5, 0.5, 'equation parse failed',
                          transform=axes[row, 2].transAxes,
                          ha='center', va='center', fontsize=11)
        axes[row, 2].set_title(f'{name} — ODEFormer', fontsize=10)

plt.tight_layout()
plt.show()

---
## 11. FIM-ODE — Context-Size Experiment

We evaluate how FIM-ODE performance scales with context size and IC diversity.
For each task (VDP-uniform, VDP-nonuniform, FHN) we fix **one reference trajectory**
from the dataset and augment it with $k-1$ additional trajectories generated under
two IC strategies:

| IC strategy | Additional ICs |
|:------------|:---------------|
| **perturbed** | drawn from $\mathcal{N}(x_0,\, 0.1\,I)$ |
| **random**    | drawn from $U([-3,3]^2)$ for VDP, $U([-2,2]^2)$ for FHN |

Context sizes tested: $k \in \{3,\, 9,\, 50\}$.
Each condition is repeated **100 times** (ICs and noise resampled every repeat).


In [ ]:
import sys
from pathlib import Path
_nb = Path(globals().get('__vsc_ipynb_file__',
           __file__ if '__file__' in dir() else __import__('os').getcwd()))
_root = _nb.parent
while not (_root / 'scripts' / 'ode').exists() and _root != _root.parent:
    _root = _root.parent
if str(_root / 'scripts' / 'ode') not in sys.path:
    sys.path.insert(0, str(_root / 'scripts' / 'ode'))

import importlib, fim_ode
importlib.reload(fim_ode)
from fim_ode import (
    vdp_dynamics, fhn_dynamics,
    generate_ensemble, split_fhn_trajectories,
    predict_and_integrate_ode, make_fim_ode_fn,
    mse_stats,
)
import numpy as np, torch, time
from scipy.integrate import solve_ivp

N_REPEATS    = 100
SEED         = 42
K_EXTRA_LIST = (2, 8, 49)   # context totals: 3, 9, 50

# ── Build ensembles ────────────────────────────────────────────────────────────
# Each ensemble: dict with keys "perturbed_k{3,9,50}" and "random_k{3,9,50}",
# each array (N_REPEATS, k_total, T, D) where [:, 0] = fixed reference trajectory.

print('Building ensembles...')

ens_vdp_u = generate_ensemble(
    vdp_dynamics,
    ref_traj  = ref_vdp_u['train_ys'][0],          # (T_train, D)
    ref_times = ref_vdp_u['train_ts'],
    y0        = ref_vdp_u['x0'],
    noise_var = float(ref_vdp_u['obs_noise_var']),
    x0_bounds = (-3., 3.),
    n_repeats=N_REPEATS, k_extra_list=K_EXTRA_LIST, seed=SEED,
)
print('  VDP-u done')

ens_vdp_n = generate_ensemble(
    vdp_dynamics,
    ref_traj  = ref_vdp_n['train_ys'][0],          # (T_train, D)
    ref_times = ref_vdp_n['train_ts'],
    y0        = ref_vdp_n['x0'],
    noise_var = float(ref_vdp_n['obs_noise_var']),
    x0_bounds = (-3., 3.),
    n_repeats=N_REPEATS, k_extra_list=K_EXTRA_LIST, seed=SEED,
)
print('  VDP-nu done')

ens_fhn = generate_ensemble(
    fhn_dynamics,
    ref_traj  = ref_fhn['full_ys'][0],             # (T_full, D) raw — split inside eval loop
    ref_times = ref_fhn['full_ts'],
    y0        = ref_fhn['x0'],
    noise_var = float(ref_fhn['obs_noise_var']),
    x0_bounds = (-2., 2.),
    n_repeats=N_REPEATS, k_extra_list=K_EXTRA_LIST, seed=SEED,
)
print('  FHN done')

# ── VDP evaluation ─────────────────────────────────────────────────────────────
def _eval_vdp(ens, y0, train_ts, test_ts, test_ys, task_label):
    full_ts       = np.sort(np.concatenate([train_ts, test_ts]))
    context_split = len(train_ts)
    results       = {}

    for key, data in ens.items():
        if not key.startswith(('perturbed', 'random')):
            continue
        t0 = time.time()
        print(f'  {task_label:12s}  {key:16s}  ×{N_REPEATS} ...', end=' ', flush=True)
        errs = []
        for r in range(N_REPEATS):
            sol = predict_and_integrate_ode(
                model, data[r], train_ts, y0, full_ts, device)
            errs.append(float(np.mean(
                (sol.y.T[context_split:] - test_ys[0]) ** 2)))
        results[key] = mse_stats(errs)
        print(f'done ({time.time()-t0:.1f}s)')
    return results


# ── FHN evaluation ─────────────────────────────────────────────────────────────
def _eval_fhn(ens, y0, full_ts, miss_mask, target_ys, task_label):
    results = {}

    for key, data in ens.items():
        if not key.startswith(('perturbed', 'random')):
            continue
        t0 = time.time()
        print(f'  {task_label:12s}  {key:16s}  ×{N_REPEATS} ...', end=' ', flush=True)
        errs, skipped = [], 0
        for r in range(N_REPEATS):
            # split_fhn_trajectories handles all k_total trajectories together,
            # padding to a common max_len automatically
            ctx_trajs, ctx_times, ctx_mask = split_fhn_trajectories(
                data[r], full_ts)
            if ctx_trajs is None:
                skipped += 1
                continue
            traj_t = torch.tensor(ctx_trajs, dtype=torch.float32, device=device).unsqueeze(0)
            time_t = torch.tensor(ctx_times, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(-1)
            mask_t = torch.tensor(ctx_mask,  dtype=torch.bool,    device=device).unsqueeze(0).unsqueeze(-1)
            fim_fn = make_fim_ode_fn(model, traj_t, time_t, mask_t)
            sol    = solve_ivp(
                fim_fn,
                t_span=(float(full_ts[0]), float(full_ts[-1])),
                y0=np.asarray(y0, dtype=float),
                t_eval=full_ts, method='RK45', rtol=1e-4, atol=1e-6,
            )
            errs.append(float(np.mean((sol.y.T[miss_mask] - target_ys) ** 2)))
        results[key] = mse_stats(errs)
        suffix = f'  [{skipped} skipped]' if skipped else ''
        print(f'done ({time.time()-t0:.1f}s){suffix}')
    return results


# ── Run ────────────────────────────────────────────────────────────────────────
print('\n=== FIM-ODE — context-size experiment ===')
print('\n[VDP-uniform]')
fim_vdp_u = _eval_vdp(ens_vdp_u, ref_vdp_u['x0'],
                       ref_vdp_u['train_ts'], ref_vdp_u_test_ts, ref_vdp_u_test_ys,
                       'VDP-u')

print('\n[VDP-nonuniform]')
fim_vdp_n = _eval_vdp(ens_vdp_n, ref_vdp_n['x0'],
                       ref_vdp_n['train_ts'], ref_vdp_n_test_ts, ref_vdp_n_test_ys,
                       'VDP-nu')

print('\n[FHN]')
fim_fhn = _eval_fhn(ens_fhn, ref_fhn['x0'],
                    ref_fhn['full_ts'], fhn_ref_miss_mask, fhn_ref_target_ys,
                    'FHN')

# ── Summary table ─────────────────────────────────────────────────────────────
IC_MODES = ('perturbed', 'random')
K_TOTALS = [1 + k for k in K_EXTRA_LIST]    # [3, 9, 50]

hdr = (f'  {"task":<14} {"ic_mode":<10} {"k":>4}  '
       f'{"mean":>8} {"std":>8} {"median":>8} {"q05":>8} {"q95":>8}')
print('\n' + '=' * len(hdr))
print(hdr)
print('=' * len(hdr))
for task_label, res in [('VDP-uniform', fim_vdp_u),
                         ('VDP-nonuniform', fim_vdp_n),
                         ('FHN', fim_fhn)]:
    for ic_mode in IC_MODES:
        for k in K_TOTALS:
            key = f'{ic_mode}_k{k}'
            if key not in res:
                continue
            s = res[key]
            print(f'  {task_label:<14} {ic_mode:<10} {k:>4}  '
                  f'{s["mean"]:>8.4f} {s["std"]:>8.4f} '
                  f'{s["median"]:>8.4f} {s["q05"]:>8.4f} {s["q95"]:>8.4f}')
    print('-' * len(hdr))


---
## 12. MoCap Dataset Exploration

Three motion-capture subjects from the **CMU MoCap** database (originally used in the UAGP-ODE paper):

| Subject | Activities (train / val / test) | Raw dim | PCA dim | dt |
|---------|--------------------------------|---------|---------|-----|
| 09      | 6 / 2 / 2                      | 50      | 5       | 0.01 s |
| 35      | 16 / 3 / 4                     | 50      | 5       | 0.01 s |
| 39      | 6 / 2 / 2                      | 50      | 5       | 0.01 s |

**Data pipeline:**
1. Raw 50D joint-angle time series are loaded from `data/mocap/mocap{sub}.npz`
2. A 5-component PCA is fitted on the training set (all activities, all time steps)
3. PCA projections are normalised to unit std per component
4. The result is stored in `subject_{sub}/{short,long}/mocap_dataset.pkl`
5. The `obs_values.h5` / `obs_times.h5` / `obs_mask.h5` / `locations.h5` files are derived from the pkl for use as direct FIM-ODE input

This section inspects the data at each stage of that pipeline.

### 12.1 Load datasets

In [ ]:
import sys, pickle, importlib
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

# data_gen_mocap lives in scripts/ode/ — already on sys.path from cell 2
import data_gen_mocap as _dgm
importlib.reload(_dgm)
from data_gen_mocap import MocapDataset, Normalize, Data

# ── Pickle compatibility patch ────────────────────────────────────────────────
# The pkl files were saved with Normalize/Data/MocapDataset in __main__.
# Expose them there so pickle.load can resolve the class references.
import __main__
__main__.Normalize    = Normalize
__main__.Data         = Data
__main__.MocapDataset = MocapDataset

DATA_DIR = _repo_root / 'data' / 'mocap'
SUBJECTS  = ['09', '35', '39']
LENGTHS   = ['short', 'long']

# ── Load pkl datasets ─────────────────────────────────────────────────────────
datasets = {}   # datasets[sub][length] = MocapDataset
for sub in SUBJECTS:
    datasets[sub] = {}
    for length in LENGTHS:
        pkl_path = DATA_DIR / f'subject_{sub}' / length / 'mocap_dataset.pkl'
        datasets[sub][length] = MocapDataset.load_from_pickle(pkl_path)

# ── Load raw NPZ files ────────────────────────────────────────────────────────
raw = {}   # raw[sub] = dict with 'train'/'validation'/'test' arrays (n, T, 50)
for sub in SUBJECTS:
    raw[sub] = dict(np.load(DATA_DIR / f'mocap{sub}.npz'))

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'{"Subject":<10} {"length":<7} {"trn (n,T,D)":<18} {"val (n,T,D)":<18} '
      f'{"tst (n,T,D)":<18} {"seqlen":<8} {"PCA":<5}')
print('-' * 85)
for sub in SUBJECTS:
    for length in LENGTHS:
        ds = datasets[sub][length]
        print(f'  {sub:<8} {length:<7} '
              f'{str(ds.trn.ys.shape):<18} '
              f'{str(ds.val.ys.shape):<18} '
              f'{str(ds.tst.ys.shape):<18} '
              f'{ds.seqlen:<8} {ds.pca_components}')


### 12.2 PCA explained variance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=False)
fig.suptitle('PCA explained variance (5 components, fit on training set)', fontsize=12)

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

for ax, sub in zip(axes, SUBJECTS):
    pca   = datasets[sub]['short'].pca     # same PCA for short and long
    ev    = pca.explained_variance_ratio_
    cumev = ev.cumsum()
    n     = len(ev)
    x     = np.arange(1, n + 1)

    bars = ax.bar(x, ev, color=colors[:n], alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.plot(x, cumev, 'k--o', ms=5, lw=1.5, label='cumulative')

    for i, (bar, val, cum) in enumerate(zip(bars, ev, cumev)):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
                f'{val:.1%}', ha='center', va='bottom', fontsize=7.5)
    ax.text(n, cumev[-1] - 0.04, f'{cumev[-1]:.1%}', ha='right', va='top',
            fontsize=8, color='k')

    ax.set_xticks(x)
    ax.set_xticklabels([f'PC{i}' for i in x])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Explained variance ratio')
    ax.set_title(f'Subject {sub}')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ── Print normalization std per component ────────────────────────────────────
print('\nPCA normalisation std (unit = PCA score units):')
print(f'  {"Subject":<10} {"PC1":>8} {"PC2":>8} {"PC3":>8} {"PC4":>8} {"PC5":>8}')
for sub in SUBJECTS:
    s = datasets[sub]['short'].pca_normalize.std[0, 0]
    print(f'  {sub:<10}' + ''.join(f'{v:>8.2f}' for v in s))

### 12.3 Trajectories in PCA space

Each row = one subject. Columns = three 2D projections of the 5D PCA space.
Train trajectories are plotted in colour; val/test are shown in grey for reference.
Dots mark the start of each trajectory.

In [ ]:
PROJECTIONS = [(0, 1), (0, 2), (1, 2)]   # (PCa, PCb) pairs
PROJ_LABELS = ['PC1 vs PC2', 'PC1 vs PC3', 'PC2 vs PC3']
CMAP        = plt.cm.tab10

fig, axes = plt.subplots(len(SUBJECTS), len(PROJECTIONS),
                         figsize=(13, 4 * len(SUBJECTS)))
fig.suptitle('Trajectories in PCA space — train (colour) | val+test (grey)',
             fontsize=12)

for row, sub in enumerate(SUBJECTS):
    # Use 'long' for a richer view: longer time context, same PCA
    ds   = datasets[sub]['long']
    trn  = ds.trn.ys   # (n_trn, seqlen, 5)
    val  = ds.val.ys   # (n_val, T_val, 5)
    tst  = ds.tst.ys   # (n_tst, T_tst, 5)

    for col, ((pa, pb), plabel) in enumerate(zip(PROJECTIONS, PROJ_LABELS)):
        ax = axes[row, col]

        # val + test in grey background
        for trajs, ls in [(val, '--'), (tst, ':')]:
            for traj in trajs:
                ax.plot(traj[:, pa], traj[:, pb], color='#aaaaaa',
                        lw=0.8, ls=ls, alpha=0.6)

        # train in colour
        for i, traj in enumerate(trn):
            c = CMAP(i / max(len(trn) - 1, 1))
            ax.plot(traj[:, pa], traj[:, pb], color=c, lw=1.2, alpha=0.85)
            ax.plot(traj[0, pa], traj[0, pb], 'o', color=c, ms=5, zorder=5)

        ax.set_xlabel(f'PC{pa+1}')
        ax.set_ylabel(f'PC{pb+1}')
        ax.set_title(f'Subject {sub} — {plabel}')
        ax.grid(alpha=0.2)

# Legend outside
handles = [plt.Line2D([0], [0], color='#aaaaaa', ls='--', lw=1.2, label='val'),
           plt.Line2D([0], [0], color='#aaaaaa', ls=':', lw=1.2, label='test'),
           plt.Line2D([0], [0], color=CMAP(0), lw=1.5, label='train'),
           plt.Line2D([0], [0], marker='o', color='grey', ls='none', ms=6,
                      label='start')]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

### 12.4 Raw joint angles vs PCA reconstruction

Verify that 5 PCA components faithfully reconstruct the 50D raw signal.
One representative training trajectory per subject is shown, with the first 6 joints plotted.
The shaded band is the residual |raw − reconstructed|.

In [ ]:
N_JOINTS_SHOW = 6   # how many of the 50 joints to overlay

fig, axes = plt.subplots(len(SUBJECTS), 2,
                         figsize=(13, 3.5 * len(SUBJECTS)),
                         gridspec_kw={'width_ratios': [3, 1]})
fig.suptitle('Raw 50D signal vs 5-component PCA reconstruction', fontsize=12)

for row, sub in enumerate(SUBJECTS):
    ds  = datasets[sub]['short']
    pca = ds.pca

    # ── Pick first training trajectory in raw space ───────────────────
    raw_traj = raw[sub]['train'][0]              # (T_raw, 50)
    T_raw    = raw_traj.shape[0]
    t_raw    = ds.dt * np.arange(T_raw)

    # PCA uses training data mean; treat zero dimensions consistently
    raw_clean = raw_traj.copy()
    raw_clean[:, [24, 25, 31, 32]] = 1e-6       # same zero-treatment as MocapDataset

    # project → reconstruct
    proj  = pca.transform(raw_clean)             # (T_raw, 5)
    recon = pca.inverse_transform(proj)          # (T_raw, 50)

    residual_rms = np.sqrt(np.mean((raw_clean - recon) ** 2, axis=1))  # (T_raw,)

    # ── Left: time-series overlay ─────────────────────────────────────
    ax = axes[row, 0]
    for j in range(N_JOINTS_SHOW):
        c = CMAP(j / N_JOINTS_SHOW)
        ax.plot(t_raw, raw_clean[:, j],  color=c, lw=1.4, ls='-',
                label=f'joint {j}' if row == 0 else None)
        ax.plot(t_raw, recon[:, j],      color=c, lw=1.0, ls='--', alpha=0.7)
    # residual as shaded band
    ax2 = ax.twinx()
    ax2.fill_between(t_raw, 0, residual_rms, color='red', alpha=0.15)
    ax2.set_ylabel('RMS residual', color='red', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='red', labelsize=7)
    ax2.set_ylim(0, residual_rms.max() * 4)

    ax.set_xlabel('t (s)')
    ax.set_ylabel('Joint angle (raw units)')
    ax.set_title(f'Subject {sub} — raw (solid) vs recon (dashed), first {N_JOINTS_SHOW} joints')
    ax.grid(alpha=0.2)
    if row == 0:
        ax.legend(fontsize=7, ncol=3, loc='upper right')

    # ── Right: per-component variance bar ─────────────────────────────
    ax = axes[row, 1]
    ev  = pca.explained_variance_ratio_
    x   = np.arange(1, len(ev) + 1)
    ax.barh(x, ev * 100, color=colors[:len(ev)], alpha=0.85)
    ax.barh(x, [0], color='none')   # dummy for spacing
    for xi, val in zip(x, ev):
        ax.text(val * 100 + 0.5, xi, f'{val:.1%}', va='center', fontsize=8)
    ax.set_xlabel('Variance explained (%)')
    ax.set_yticks(x)
    ax.set_yticklabels([f'PC{i}' for i in x])
    ax.set_title(f'Subject {sub}')
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

# ── Reconstruction MSE summary ────────────────────────────────────────────────
print('\nReconstruction RMSE (mean over time, averaged over all training trajectories):')
for sub in SUBJECTS:
    ds  = datasets[sub]['short']
    pca = ds.pca
    all_raw = raw[sub]['train'].copy()
    all_raw[:, :, [24, 25, 31, 32]] = 1e-6
    proj  = pca.transform(all_raw.reshape(-1, 50))
    recon = pca.inverse_transform(proj).reshape(all_raw.shape)
    rmse  = float(np.sqrt(np.mean((all_raw - recon) ** 2)))
    print(f'  Subject {sub}: RMSE = {rmse:.4f}')

### 12.5 FIM-ODE input format (h5 files)

The `obs_values.h5` / `obs_times.h5` / `obs_mask.h5` / `locations.h5` files are the
FIM-ODE-ready tensors derived from the pkl. This cell reads them and cross-checks against
the pkl to confirm consistency.

In [ ]:
print('FIM-ODE h5 tensor shapes and consistency check\n')
print(f'  {"Subject/length":<20} {"obs_values":<22} {"obs_times":<22} '
      f'{"obs_mask":<22} {"locations":<20}')
print('-' * 92)

for sub in SUBJECTS:
    for length in LENGTHS:
        base = DATA_DIR / f'subject_{sub}' / length
        shapes = {}
        for fname in ['obs_values', 'obs_times', 'obs_mask', 'locations']:
            with h5py.File(base / f'{fname}.h5', 'r') as hf:
                shapes[fname] = tuple(hf['data'].shape)
        tag = f'{sub}/{length}'
        print(f'  {tag:<20} {str(shapes["obs_values"]):<22} {str(shapes["obs_times"]):<22} '
              f'{str(shapes["obs_mask"]):<22} {str(shapes["locations"]):<20}')

# ── Cross-check: obs_values[0] == pkl trn.ys for short ─────────────────────
print('\nCross-check: obs_values.h5 matches pkl trn.ys')
for sub in SUBJECTS:
    base = DATA_DIR / f'subject_{sub}' / 'short'
    with h5py.File(base / 'obs_values.h5', 'r') as hf:
        h5_vals = hf['data'][0]    # (n_trn, seqlen, 5)
    pkl_vals = datasets[sub]['short'].trn.ys   # (n_trn, seqlen, 5)
    max_diff = float(np.abs(h5_vals - pkl_vals).max())
    print(f'  Subject {sub}: max|h5 - pkl.trn.ys| = {max_diff:.2e}  '
          f'{"✓ match" if max_diff < 1e-4 else "✗ MISMATCH"}')

# ── locations.h5 = [start, end] of each training trajectory ─────────────────
print('\nlocations.h5 = [start_point, end_point] check for subject_09/short:')
sub = '09'
base = DATA_DIR / f'subject_{sub}' / 'short'
with h5py.File(base / 'locations.h5', 'r') as hf:
    locs = hf['data'][()]   # (n_trn, 2, 5)
pkl_trn = datasets[sub]['short'].trn.ys
print(f'  locations shape: {locs.shape}  — interpreted as (n_traj, 2, 5)')
print(f'  locs[0, 0] (start):  {np.round(locs[0, 0], 3)}')
print(f'  trn.ys[0, 0]:        {np.round(pkl_trn[0, 0], 3)}')
print(f'  locs[0, 1] (end?):   {np.round(locs[0, 1], 3)}')
print(f'  trn.ys[0, 1] (t=1):  {np.round(pkl_trn[0, 1], 3)}')

### 12.6 Zero-shot evaluation in 50D joint-angle space

FIM-ODE was trained on systems up to dimension 3.  We therefore feed it
only the **first 3 PCA components** of each MoCap training trajectory.
Predicted 3-component trajectories are padded back to 5 dimensions (zeros
for components 4–5), then inverted through `pca_normalize` and `pca` to
recover 50D joint-angle predictions.  MSE is measured in that 50D space so
it reflects the full reconstruction error, including the variance carried
by the un-modelled components.

In [ ]:
import importlib, fim_ode
importlib.reload(fim_ode)
from fim_ode import eval_mocap_zero_shot

N_PCA = 3   # FIM-ODE supports up to dim 3

print('=== FIM-ODE — MoCap zero-shot evaluation (50D MSE) ===\n')
print(f'  {"Subject/variant":<22} {"n_trn":>6} {"T_trn":>6} '
      f'{"n_tst":>6} {"T_tst":>6} {"MSE (50D)":>12}')
print('  ' + '-' * 62)

mocap_results = {}
for sub in SUBJECTS:
    for length in LENGTHS:
        ds    = datasets[sub][length]
        label = f'{sub}/{length}'

        trn_ys = np.asarray(ds.trn.ys)
        tst_ys = np.asarray(ds.tst.ys)

        preds_50d, mse = eval_mocap_zero_shot(
            model, ds, device=device, n_pca_dims=N_PCA
        )

        mocap_results[label] = dict(
            preds_50d=preds_50d, mse=mse,
            n_trn=trn_ys.shape[0], T_trn=trn_ys.shape[1],
            n_tst=tst_ys.shape[0], T_tst=tst_ys.shape[1],
        )
        print(f'  {label:<22} {trn_ys.shape[0]:>6d} {trn_ys.shape[1]:>6d} '
              f'{tst_ys.shape[0]:>6d} {tst_ys.shape[1]:>6d} {mse:>12.5f}')


---
## 13. MoCap Finetuning — Evaluation

Load the best finetuned checkpoints, evaluate on the held-out **test** trajectories
in 50D joint-angle space, then release the model from memory.

| Dataset | Checkpoint | lr | n\_inner | Best epoch |
|---|---|---|---|---|
| mocap-09-short | `mocap-09-short_20260526_142845_best.pt` | 5e-6 | 5 | 70 |
| mocap-09-long  | `mocap-09-long_20260526_151151_best.pt`  | 5e-6 | 5 | 20 (still training) |


In [ ]:
import copy, gc, importlib.util
import torch

# Load utilities from fim-ode-finetune.py (hyphen prevents normal import)
_ft_path = _repo_root / 'scripts' / 'ode' / 'fim-ode-finetune.py'
_ft_spec = importlib.util.spec_from_file_location('fim_ode_finetune', _ft_path)
fim_ode_finetune = importlib.util.module_from_spec(_ft_spec)
_ft_spec.loader.exec_module(fim_ode_finetune)

load_fim_ode_hf          = fim_ode_finetune.load_fim_ode_hf
load_mocap               = fim_ode_finetune.load_mocap
make_mocap_test_eval_fn  = fim_ode_finetune.make_mocap_test_eval_fn


In [ ]:
_ckpt_dir  = _repo_root / 'scripts' / 'ode' / 'checkpoints' / 'finetune'
_mocap_dir = _repo_root / 'data' / 'mocap'

# Load base architecture once; deepcopy per checkpoint to avoid re-downloading
_base_model = load_fim_ode_hf(device=device)
_base_model.eval()

MOCAP_CHECKPOINTS = [
    ('09', 'short', _ckpt_dir / 'mocap-09-short_20260526_142845_best.pt'),  # lr=5e-6, n_inner=5
    ('09', 'long',  _ckpt_dir / 'mocap-09-long_20260526_151151_best.pt'),   # lr=5e-6, n_inner=5
]

finetune_mocap_results = {}

for subject, variant, ckpt_path in MOCAP_CHECKPOINTS:
    label = f'mocap-{subject}-{variant}'
    print(f'
── {label} ──')

    _ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    _model = copy.deepcopy(_base_model)
    _model.load_state_dict(_ckpt['model_state_dict'])
    _model.eval()

    _data    = load_mocap(_mocap_dir, subject=subject, variant=variant)
    _eval_fn = make_mocap_test_eval_fn(_data, device)
    with torch.no_grad():
        mse = _eval_fn(_model)

    finetune_mocap_results[label] = {'mse': mse, 'epoch': _ckpt['epoch']}
    print(f'  test MSE (50D, epoch {_ckpt["epoch"]}): {mse:.5f}')

    del _model, _ckpt, _data, _eval_fn
    gc.collect()

del _base_model
gc.collect()


### 13.1 Johannes's finetuned models (HuggingFace)

Download and evaluate the 6 MoCap checkpoints from `FIM4Science/fim-ode` on the held-out test set (50D MSE).

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file as _load_safetensors
import json as _json

_REPO = 'FIM4Science/fim-ode'

HF_TASKS = {
    "09_short": "mocap09short/mocap09short_01-29-0610/checkpoints/best-model",
    "09_long": "mocap09long/mocap09long_01-29-0609/checkpoints/best-model",
    "35_short": "mocap35short/mocap35short_01-29-0614/checkpoints/best-model",
    "35_long": "mocap35long/mocap35long_01-29-0615/checkpoints/best-model",
    "39_short": "mocap39short/mocap39short_01-29-0618/checkpoints/best-model",
    "39_long": "mocap39long/mocap39long_01-29-0619/checkpoints/best-model",
}

def load_mocap_finetuned_johannes(task_key: str, device: str = 'cpu'):
    subfolder = HF_TASKS[task_key]
    config_path  = hf_hub_download(_REPO, f'{subfolder}/config.json')
    weights_path = hf_hub_download(_REPO, f'{subfolder}/model.safetensors')
    with open(config_path) as f:
        config_dict = _json.load(f)
    _cfg = fim_ode_finetune.FIMODEConfig()
    _cfg.model_config = config_dict['model_config']
    _cfg.train_config = config_dict['train_config']
    _m = fim_ode_finetune.FIMODE(_cfg)
    state_dict = _load_safetensors(weights_path, device=device)
    state_dict = {k.removeprefix('model.'): v for k, v in state_dict.items()}
    _m.load_state_dict(state_dict)
    _m = _m.to(device)
    _m.eval()
    return _m

finetune_mocap_johannes = {}

for _task_key in HF_TASKS:
    _subj, _var = _task_key.split('_')
    _label = f'mocap-{_subj}-{_var}'
    print(f'── {_label} ──')
    _model = load_mocap_finetuned_johannes(_task_key, device=device)
    _data  = load_mocap(_mocap_dir, subject=_subj, variant=_var)
    _eval_fn = make_mocap_test_eval_fn(_data, device)
    with torch.no_grad():
        _mse = _eval_fn(_model)
    finetune_mocap_johannes[_label] = _mse
    print(f'  test MSE (50D): {_mse:.5f}')
    del _model, _data, _eval_fn
    gc.collect()


In [ ]:
_ckpt_dir  = _repo_root / 'scripts' / 'ode' / 'checkpoints' / 'finetune'
_mocap_dir = _repo_root / 'data' / 'mocap'

MOCAP_CHECKPOINTS = [
    ('09', 'short', _ckpt_dir / 'mocap-09-short_epoch10_best.pt'),
    ('09', 'long',  _ckpt_dir / 'mocap-09-long_epoch30_best.pt'),
    ('35', 'short', _ckpt_dir / 'mocap-35-short_epoch80_best.pt'),
]

finetune_mocap_results = {}

for subject, variant, ckpt_path in MOCAP_CHECKPOINTS:
    label = f'mocap-{subject}-{variant}'
    print(f'\n── {label} ──')

    # Load model architecture, restore finetuned weights
    _model = load_fim_ode_hf(device=device)
    _ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    _model.load_state_dict(_ckpt['model_state_dict'])
    _model.eval()

    # Evaluate on test set
    _data    = load_mocap(_mocap_dir, subject=subject, variant=variant)
    _eval_fn = make_mocap_eval_fn(_data, device)
    with torch.no_grad():
        mse = _eval_fn(_model)

    finetune_mocap_results[label] = {'mse': mse, 'epoch': _ckpt['epoch']}
    print(f'  test MSE (epoch {_ckpt["epoch"]}): {mse:.5e}')

    # Release from memory
    del _model, _ckpt, _data, _eval_fn
    gc.collect()

In [ ]:
# Comparison table: zero-shot (section 14.6) vs finetuned
print(f'\n{"Dataset":<25} {"Zero-shot MSE":>14} {"Finetuned MSE":>15} {"Improvement":>12}')
print('-' * 70)
for label, res in finetune_mocap_results.items():
    mse1  = res['mse']
    mse0  = mocap_results.get(label.replace('mocap-', '').replace('-', '/'), {}).get('mse', float('nan'))
    pct   = 100 * (mse0 - mse1) / mse0 if mse0 > 0 else float('nan')
    print(f'  {label:<23} {mse0:>14.5e} {mse1:>15.5e} {pct:>11.1f}%')

In [ ]:
import sys, json as _json, importlib.util
import torch
import matplotlib.pyplot as plt

# Add odebench/ to path so plot_strogatz (local import inside plots_for_paper) is found
_odb_dir = str(_repo_root / 'scripts' / 'ode' / 'odebench')
if _odb_dir not in sys.path:
    sys.path.insert(0, _odb_dir)

# Load plots_for_paper.py directly to avoid triggering odebench/__init__.py
_spec = importlib.util.spec_from_file_location(
    "plots_for_paper",
    _repo_root / 'scripts' / 'ode' / 'odebench' / 'plots_for_paper.py',
)

## 14. ODEBench ODE 26 — Vector Field Comparison

Side-by-side comparison of the ground-truth vector field, FIM-ODE, and ODEFormer
for ODE 26 from the Strogatz/ODEBench benchmark.  Models are already loaded from
sections 2 and 5 respectively — no re-download needed.

In [ ]:

_pfp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pfp)
plot_comparison_1x3              = _pfp.plot_comparison_1x3
plot_comparison_1x3_fixed_points = _pfp.plot_comparison_1x3_fixed_points

from utils.eval_models import FimOdeHFEval, OdeFormerEval

# ── Wrap already-loaded models — no re-download ──────────────────────────────
class _FimOdeWrapper(FimOdeHFEval):
    """Use the already-loaded FIM-ODE model instead of fetching from HuggingFace."""
    def __init__(self, fim_model, device: str = 'cpu'):
        self._fim          = fim_model
        self.device        = torch.device(device)
        self._wrapped_D    = None
        self._feature_mask = None
        self._concept      = None
        self._D_in         = None

class _OdeFormerWrapper(OdeFormerEval):
    """Use the already-loaded ODEFormer model instead of re-downloading."""
    def __init__(self, odeformer_model, device: str = 'cpu'):
        self.model                = odeformer_model
        self.device               = torch.device(device)
        self.dtype                = torch.float64
        self.f                    = None
        self.symbolic_predictions = None
        self._map                 = OdeFormerEval._torch_lambdify_map_for_odeformer()

fim_wrapped  = _FimOdeWrapper(model,      device='cpu')
odef_wrapped = _OdeFormerWrapper(odeformer, device='cpu')

# ── Load ODEBench JSON (shared across ODE 26/42/28 cells) ────────────────────
_json_path = _repo_root / 'scripts' / 'ode' / 'odebench' / 'data' / 'strogatz_extended.json'
with open(_json_path) as _f:
    _ode_by_id = {item['id']: item for item in _json.load(_f) if item.get('dim') == 2}

# ── ODE 26: LV Competition — all fixed points ────────────────────────────────
_gt_fps_26 = [(tuple(r['fp']), r['label']) for r in gt_lv]

_fim_fps_26 = [
    (tuple(fp), stability_analysis(numerical_jacobian(fim_results[26]['vf'], fp))['label'])
    for fp, _res in fim_results[26]['fps']
]

_odef_fps_26 = (
    [(tuple(fp), stability_analysis(numerical_jacobian(od_results[26]['vf'], fp))['label'])
     for fp, _res in od_results[26]['fps']]
    if od_results[26]['vf'] is not None else []
)

fig26 = plot_comparison_1x3_fixed_points(
    ode=_ode_by_id[26],
    model1=fim_wrapped,
    model2=odef_wrapped,
    fps_gt=_gt_fps_26,
    fps_model1=_fim_fps_26,
    fps_model2=_odef_fps_26,
    sigma=0.03,
    rho=0.5,
    figsize=(30, 15),
    with_colorbar=False,
)
_out26 = _repo_root / 'scripts' / 'ode' / 'odebench' / 'odebench_ode_26.pdf'
fig26.savefig(_out26, bbox_inches='tight')
plt.show()
print(f'Saved {_out26}')


In [ ]:
# ODE 42: CDIMA reaction — first fixed point only
_gt_fps_42 = [(tuple(gt_cdima[0]['fp']), gt_cdima[0]['label'])] if gt_cdima else []

_fim_fps_42 = (
    [(tuple(fim_results[42]['fps'][0][0]),
      stability_analysis(numerical_jacobian(fim_results[42]['vf'], fim_results[42]['fps'][0][0]))['label'])]
    if fim_results[42]['fps'] else []
)

_odef_fps_42 = (
    [(tuple(od_results[42]['fps'][0][0]),
      stability_analysis(numerical_jacobian(od_results[42]['vf'], od_results[42]['fps'][0][0]))['label'])]
    if od_results[42]['vf'] is not None and od_results[42]['fps'] else []
)

fig42 = plot_comparison_1x3_fixed_points(
    ode=_ode_by_id[42],
    model1=fim_wrapped,
    model2=odef_wrapped,
    fps_gt=_gt_fps_42,
    fps_model1=_fim_fps_42,
    fps_model2=_odef_fps_42,
    sigma=0.03,
    rho=0.5,
    figsize=(30, 15),
    with_colorbar=False,
)
_out42 = _repo_root / 'scripts' / 'ode' / 'odebench' / 'odebench_ode_42.pdf'
fig42.savefig(_out42, bbox_inches='tight')
plt.show()
print(f'Saved {_out42}')


In [ ]:
# ODE 28: pendulum — first fixed point only
_gt_fps_28 = [(tuple(gt_pendulum[0]['fp']), gt_pendulum[0]['label'])] if gt_pendulum else []

_fim_fps_28 = (
    [(tuple(fim_results[28]['fps'][0][0]),
      stability_analysis(numerical_jacobian(fim_results[28]['vf'], fim_results[28]['fps'][0][0]))['label'])]
    if fim_results[28]['fps'] else []
)

_odef_fps_28 = (
    [(tuple(od_results[28]['fps'][0][0]),
      stability_analysis(numerical_jacobian(od_results[28]['vf'], od_results[28]['fps'][0][0]))['label'])]
    if od_results[28]['vf'] is not None and od_results[28]['fps'] else []
)

# Rename "centre / marginal" -> "marginal center" for all three subplots
def _fix_label(fp_list):
    return [(coords, "marginal center" if lbl == "centre / marginal" else lbl)
            for coords, lbl in fp_list]

_gt_fps_28   = _fix_label(_gt_fps_28)
_fim_fps_28  = _fix_label(_fim_fps_28)
_odef_fps_28 = _fix_label(_odef_fps_28)

fig28 = plot_comparison_1x3_fixed_points(
    ode=_ode_by_id[28],
    model1=fim_wrapped,
    model2=odef_wrapped,
    fps_gt=_gt_fps_28,
    fps_model1=_fim_fps_28,
    fps_model2=_odef_fps_28,
    sigma=0.03,
    rho=0.5,
    figsize=(30, 15),
    with_colorbar=False,
)
_out28 = _repo_root / 'scripts' / 'ode' / 'odebench' / 'odebench_ode_28.pdf'
fig28.savefig(_out28, bbox_inches='tight')
plt.show()
print(f'Saved {_out28}')


---
## 15. MoCap PCA Projection Plots

Paper-quality 2D PCA projection plots for subjects 09, 35, 39 (short variant).  
Each 1x3 figure shows PC0vPC1, PC0vPC2, PC1vPC2 with:  
- **Gray streamlines**: model-inferred vector field (conditioned on training context)  
- **Black crosses**: ground-truth test trajectories  
- **Green lines**: FIM-ODE predicted trajectories from test ICs  

Models: Johannes's finetuned checkpoints from `FIM4Science/fim-ode` (HuggingFace).

In [ ]:
import importlib.util, gc
import torch
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file as _load_st
import json as _json

# Reload plots module (picks up latest changes)
_mpp_spec = importlib.util.spec_from_file_location(
    'mocap_plots_for_paper',
    _repo_root / 'scripts' / 'ode' / 'mocap' / 'plots_for_paper.py',
)
_mpp = importlib.util.module_from_spec(_mpp_spec)
_mpp_spec.loader.exec_module(_mpp)

_mocap_device = 'cpu'
_mocap_dir    = _repo_root / 'data' / 'mocap'
_REPO = 'FIM4Science/fim-ode'

_HF_SHORT = {
    '09': 'mocap09short/mocap09short_01-29-0610/checkpoints/best-model',
    '35': 'mocap35short/mocap35short_01-29-0614/checkpoints/best-model',
    '39': 'mocap39short/mocap39short_01-29-0618/checkpoints/best-model',
}

def _load_johannes_short(subj, device='cpu'):
    sf = _HF_SHORT[subj]
    cfg_path = hf_hub_download(_REPO, f'{sf}/config.json')
    wts_path = hf_hub_download(_REPO, f'{sf}/model.safetensors')
    with open(cfg_path) as _f:
        _cfg_dict = _json.load(_f)
    # fim_ode_finetune loaded in cell 46
    _cfg = fim_ode_finetune.FIMODEConfig()
    _cfg.model_config = _cfg_dict['model_config']
    _cfg.train_config = _cfg_dict['train_config']
    _m = fim_ode_finetune.FIMODE(_cfg)
    _sd = _load_st(wts_path, device=device)
    _sd = {k.removeprefix('model.'): v for k, v in _sd.items()}
    _m.load_state_dict(_sd)
    return _m.to(device).eval()

(_repo_root / 'scripts' / 'ode' / 'mocap' / 'figures').mkdir(exist_ok=True)


In [ ]:
_model_09 = _load_johannes_short('09', device=_mocap_device)
_data_09  = _mpp.load_mocap(_mocap_dir, subject='09', variant='short')
fig_09 = _mpp.plot_mocap_subject(
    _data_09, _model_09, _mocap_device,
    subject_label='Subject 09 – short',
    n_test=2,
    save_path=_repo_root / 'scripts' / 'ode' / 'mocap' / 'figures' / 'mocap_09_short_pca.pdf',
)
del _model_09, _data_09; gc.collect()
plt.show()

In [ ]:
_model_35 = _load_johannes_short('35', device=_mocap_device)
_data_35  = _mpp.load_mocap(_mocap_dir, subject='35', variant='short')
fig_35 = _mpp.plot_mocap_subject(
    _data_35, _model_35, _mocap_device,
    subject_label='Subject 35 – short',
    n_test=2,
    save_path=_repo_root / 'scripts' / 'ode' / 'mocap' / 'figures' / 'mocap_35_short_pca.pdf',
)
del _model_35, _data_35; gc.collect()
plt.show()

In [ ]:
_model_39 = _load_johannes_short('39', device=_mocap_device)
_data_39  = _mpp.load_mocap(_mocap_dir, subject='39', variant='short')
fig_39 = _mpp.plot_mocap_subject(
    _data_39, _model_39, _mocap_device,
    subject_label='Subject 39 – short',
    n_test=2,
    save_path=_repo_root / 'scripts' / 'ode' / 'mocap' / 'figures' / 'mocap_39_short_pca.pdf',
)
del _model_39, _data_39; gc.collect()
plt.show()